# Run mini-infer Pallas TPU kernels on a free Colab TPU

Needs only a Google account (no identity verification, no card).

1. **Runtime -> Change runtime type -> TPU**.
2. **Runtime -> Run all**.

On the first run the cell aligns Colab's `libtpu` with its `jaxlib` (Colab's
preinstalled `libtpu` can be older than `jaxlib` and reject the compiled
kernel with 'Unsupported version: expected <= 7 but got 8'). It then restarts
the runtime once. **When it reconnects, run the cell again** and it proceeds.

It clones the public `tpu-pallas-backend` branch and runs every kernel (dense,
paged decode, paged prefill; MHA + GQA) with `interpret=False`, checking each
against a NumPy reference. A real TPU run prints `devices: [TpuDevice(...)]`
and ends with `ALL PASS`; it refuses to fall back to CPU, so green means it
genuinely ran on the TPU.

In [ ]:
import importlib.metadata as _md
import os
import subprocess
import sys

# Colab's preinstalled libtpu can lag jaxlib and reject the compiled Mosaic
# module ("Unsupported version: expected <= 7 but got 8"). Reinstall a matched
# jax[tpu] set, then restart once so the new native libtpu actually loads.
_SENTINEL = "/content/.jax_tpu_aligned"
if not os.path.exists(_SENTINEL):
    try:
        _pin = "jax[tpu]==" + _md.version("jax")
    except Exception:
        _pin = "jax[tpu]"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            _pin,
            "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ],
        check=True,
    )
    open(_SENTINEL, "w").close()
    print("Aligned libtpu with jaxlib and restarting the runtime.")
    print(">>> When the runtime reconnects, run this cell again. <<<")
    os.kill(os.getpid(), 9)  # force a runtime restart so the new libtpu loads

import jax  # noqa: E402  (after the libtpu-alignment restart guard, by design)

print("jax", jax.__version__, "devices:", jax.devices())
if not any(getattr(d, "platform", "") == "tpu" for d in jax.devices()):
    raise SystemExit("No TPU. Runtime -> Change runtime type -> TPU, then run this cell again.")

DEST = "/content/mini-infer"
BRANCH = "tpu-pallas-backend"
if os.path.isdir(DEST):
    # Colab VMs persist across runs; a clone-once bootstrap silently runs
    # stale code. Always sync the checkout to the branch head.
    subprocess.run(["git", "-C", DEST, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "https://github.com/JonathanBerhe/mini-infer.git",
            DEST,
        ],
        check=True,
    )
print(
    "running commit:",
    subprocess.run(
        ["git", "-C", DEST, "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip(),
)
sys.path.insert(0, os.path.join(DEST, "src"))
sys.path.insert(0, os.path.join(DEST, "scripts"))
os.chdir(DEST)

# Re-running the cell in a live runtime keeps old modules cached; purge ours
# so the freshly synced code is what actually runs.
_OURS = ("mini_infer", "run_tpu_pallas_kernels")
for _m in [m for m in list(sys.modules) if m.split(".")[0] in _OURS]:
    del sys.modules[_m]

import run_tpu_pallas_kernels as runner  # noqa: E402  (needs the sys.path setup above)

_rc = runner.main()
# Fail the cell on any parity FAIL so Run all cannot end green on a bad run.
assert _rc == 0, f"kernel validation FAILED (exit code {_rc})"
print("exit code:", _rc)